In [ ]:
# %%
import os
import random
from dataclasses import dataclass, field
from typing import Tuple

import torch
import torch.nn as nn
from torch.distributions import Bernoulli
from torch.optim import AdamW
from transformers import AutoTokenizer
from downstream_utils import PetsPreferenceEnv, evaluate
from peft import PeftModel
import numpy as np

class DirectPairPolicy(nn.Module):
    def __init__(self, pretrained_vae, tokenizer_name="gpt2", device=None, temperature=1.0):
        super().__init__()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.vae = pretrained_vae
        self.temperature = temperature

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        self.tokenizer.padding_side = "right"

        # 冻结 LLM encoder
        for p in self.vae.llm_encoder.parameters():
            p.requires_grad = False
        self.vae.llm_encoder.eval()

        llm_param = next(self.vae.llm_encoder.parameters())
        self.model_device = llm_param.device
        self.model_dtype = llm_param.dtype

        hidden_dim = self.vae.latent_dim

        # 复用 pair_encoder，把 (x_emb, y_emb) -> pair_hidden
        self.pair_encoder = self.vae.pair_encoder
        for p in self.pair_encoder.parameters():
            p.requires_grad = True

        pair_param = next(self.pair_encoder.parameters())
        self.model_device = pair_param.device
        self.model_dtype = pair_param.dtype

        # 一个简单 policy head：pair_hidden -> scalar logit
        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        ).to(device=self.model_device, dtype=self.model_dtype)

        self.to(self.model_device)

    @torch.no_grad()
    def _embed_text(self, text: str) -> torch.Tensor:
        tokens = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )
        tokens = {k: v.to(self.model_device) for k, v in tokens.items()}

        emb = self.vae.llm_encoder(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
        )[0]

        if emb.dim() == 1:
            emb = emb.unsqueeze(0)

        emb = emb.to(device=self.model_device, dtype=self.model_dtype)
        return emb

    def forward(self, x: str, y: str):
        with torch.no_grad():
            x_emb = self._embed_text(x)
            y_emb = self._embed_text(y)

        x_emb = x_emb.to(device=self.model_device, dtype=self.model_dtype)
        y_emb = y_emb.to(device=self.model_device, dtype=self.model_dtype)

        pair_hidden = self.pair_encoder(x_emb, y_emb)
        pair_hidden = pair_hidden.to(device=self.model_device, dtype=self.model_dtype)

        logits = self.policy_head(pair_hidden) / self.temperature
        return {
            "x_emb": x_emb,
            "y_emb": y_emb,
            "pair_hidden": pair_hidden,
            "logits": logits,
        }

    def act(self, pair: Tuple[str, str], sample=True):
        x, y = pair
        out = self.forward(x, y)

        logits = out["logits"]
        action_dist = Bernoulli(logits=logits)

        if sample:
            action = action_dist.sample()
        else:
            action = (torch.sigmoid(logits) > 0.5).to(logits.dtype)

        return {
            **out,
            "action_dist": action_dist,
            "action": int(action.item()),
            "action_tensor": action,
            "prob_choose_x": torch.sigmoid(logits).item(),
        }
    
    
class BaseRewardModel(nn.Module):
    """
    用 train_llm_preference_model 训练得到的 reward model
    输入 (x,y)，输出 preferred action
    """

    def __init__(self, model_path, tokenizer_name="gpt2", device=None):
        super().__init__()
        from transformers import AutoModelForSequenceClassification

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        base_model = AutoModelForSequenceClassification.from_pretrained(
            "gpt2",  # 必须和训练时一致
            num_labels=1,
        ).to(device)

        self.model = PeftModel.from_pretrained(
            base_model,
            model_path,
        ).to(device)

        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

    @torch.no_grad()
    def score(self, text: str):
        tokens = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )
        tokens = {k: v.to(self.device) for k, v in tokens.items()}

        out = self.model(**tokens)
        return out.logits.squeeze(-1)  # scalar

    @torch.no_grad()
    def preferred_action(self, x: str, y: str):
        score_x = self.score(x)
        score_y = self.score(y)
        return 1 if score_x > score_y else 0, score_x, score_y 

@dataclass
class BaselineScriptArguments:
    vae_model_path: str = field(default="logs/gpt2_simple_pets/both/vae_gpt2__0_0.0003_cosine_2_0.0001_512_768_seed0_peft_last_checkpoint/model.pt")
    tokenizer_name: str = field(default="gpt2")
    data_path: str = field(default="data/simple_pets/gpt2")
    data_subset: str = field(default="both")
    learning_rate: float = field(default=1e-4)
    num_steps: int = field(default=5000)
    entropy_coef: float = field(default=1)
    baseline_momentum: float = field(default=0.9)
    temperature: float = field(default=1.0)
    seed: int = field(default=0)
    eval_every: int = field(default=200)
    save_dir: str = field(default="logs/direct_pair_policy")
    use_unimodal: bool = field(default=False)
    base_model_path: str = field(default="logs/gpt2_simple_pets/both/base_gpt2__0_0.0001_cosine_2_seed0_peft_last_checkpoint")
    reward_weight: float = field(default=0.75) 

In [ ]:
# %%
args = BaselineScriptArguments(use_unimodal=True)

random.seed(args.seed)
torch.manual_seed(args.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"




In [ ]:
from collections import defaultdict
import numpy as np
import os
import random
import torch
from torch.optim import AdamW

def run(env_id, reward_weight, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    pretrained_vae = torch.load(args.vae_model_path, map_location=device, weights_only=False)
    pretrained_vae.to(device)
    pretrained_vae.eval()

    reward_model = None
    if args.use_unimodal:
        reward_model = BaseRewardModel(
            model_path=args.base_model_path,
            tokenizer_name=args.tokenizer_name,
            device=device,
        )

    for p in pretrained_vae.parameters():
        p.requires_grad = False

    policy = DirectPairPolicy(
        pretrained_vae=pretrained_vae,
        tokenizer_name=args.tokenizer_name,
        device=device,
        temperature=args.temperature,
    )

    trainable_params = [p for p in policy.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=args.learning_rate)

    pref_type = env_id
    train_env = PetsPreferenceEnv(seed=seed, pref_type=pref_type)
    test_env = PetsPreferenceEnv(seed=seed + 1, pref_type=pref_type, eval=True)

    reward_baseline = 0.0
    state = train_env.reset()

    os.makedirs(args.save_dir, exist_ok=True)
    best_acc = -1.0
    final_acc, final_cost = None, None

    for step in range(1, args.num_steps + 1):
        out = policy.act(state, sample=True)
        next_state, env_reward, done, info = train_env.step(out["action"])

        if args.use_unimodal:
            x, y = state
            rm_action, score_x, score_y = reward_model.preferred_action(x, y)
            rm_reward = 1 if out["action"] == rm_action else 0
            reward = (1 - reward_weight) * env_reward + reward_weight * rm_reward
        else:
            rm_reward = 0
            reward = env_reward

        reward_t = torch.tensor(float(reward), device=device)

        reward_baseline = (
            args.baseline_momentum * reward_baseline
            + (1 - args.baseline_momentum) * reward
        )
        advantage = reward_t - reward_baseline

        log_prob_action = out["action_dist"].log_prob(out["action_tensor"]).mean()
        entropy = out["action_dist"].entropy().mean()

        loss = -(advantage.detach() * log_prob_action) - args.entropy_coef * entropy

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()

        state = next_state

        if step % 50 == 0:
            print(
                f"seed={seed} env_id={env_id} w={reward_weight} step={step} "
                f"reward={reward:.3f} baseline={reward_baseline:.3f} "
                f"adv={float(advantage):.3f} loss={float(loss):.4f} "
                f"entropy={float(entropy):.4f} p_choose_x={out['prob_choose_x']:.4f} "
                f"env_r={env_reward:.2f} rm_r={rm_reward:.2f} final_r={reward:.2f}"
            )

        if step % args.eval_every == 0 or step == args.num_steps:
            res = evaluate(policy, test_env, num_episodes=600)
            acc = res["overall_accuracy"]
            avg_cost = res["overall_avg_cost"]
            print(
                f"[eval] seed={seed} env_id={env_id} w={reward_weight} "
                f"step={step}, test_acc={acc:.4f}, test_avg_cost={avg_cost:.4f}"
            )

            if step == args.num_steps:
                final_acc, final_cost = acc, avg_cost

            if acc > best_acc:
                best_acc = acc
                ckpt_path = os.path.join(args.save_dir, f"best_direct_pair_policy_seed{seed}_env{env_id}_w{reward_weight}.pt")
                torch.save(
                    {
                        "pair_encoder": policy.pair_encoder.state_dict(),
                        "policy_head": policy.policy_head.state_dict(),
                        "step": step,
                        "best_acc": best_acc,
                        "seed": seed,
                        "env_id": env_id,
                        "reward_weight": reward_weight,
                    },
                    ckpt_path,
                )
                print(f"saved best checkpoint to {ckpt_path}")

    return {
        "acc": float(final_acc),
        "cost": float(final_cost),
    }


def summarize_results(all_results):
    """
    all_results[w] = {
        "seed_results": [
            {"seed": s, "env0": {...}, "env1": {...}, "avg_acc": ..., "avg_cost": ...},
            ...
        ]
    }
    """
    for w, payload in all_results.items():
        seed_avgs_acc = [x["avg_acc"] for x in payload["seed_results"]]
        seed_avgs_cost = [x["avg_cost"] for x in payload["seed_results"]]

        mean_acc = float(np.mean(seed_avgs_acc))
        std_acc = float(np.std(seed_avgs_acc))
        mean_cost = float(np.mean(seed_avgs_cost))
        std_cost = float(np.std(seed_avgs_cost))

        name = "Task-Only baseline" if w == 0 else f"RC (omega={w})"

        print("\n" + "=" * 80)
        print(name)
        print(f"Across 3 seeds:")
        print(f"  avg_reward = {mean_acc:.4f} +- {std_acc:.4f}")
        print(f"  avg_cost   = {mean_cost:.4f} +- {std_cost:.4f}")
        print("=" * 80)


In [ ]:


reward_weights = [0, 0.25, 0.75]
seeds = [1, 2, 3]

all_results = defaultdict(lambda: {"seed_results": []})

for w in reward_weights:
    print(f"\n\n###############################")
    print(f"Running reward_weight = {w}")
    print(f"###############################")

    for seed in seeds:
        env0_res = run(env_id=0, reward_weight=w, seed=seed)
        env1_res = run(env_id=1, reward_weight=w, seed=seed)

        avg_acc = 0.5 * (env0_res["acc"] + env1_res["acc"])
        avg_cost = 0.5 * (env0_res["cost"] + env1_res["cost"])

        all_results[w]["seed_results"].append({
            "seed": seed,
            "env0": env0_res,
            "env1": env1_res,
            "avg_acc": avg_acc,
            "avg_cost": avg_cost,
        })

        print(
            f"[seed summary] w={w}, seed={seed}: "
            f"env0(acc={env0_res['acc']:.4f}, cost={env0_res['cost']:.4f}), "
            f"env1(acc={env1_res['acc']:.4f}, cost={env1_res['cost']:.4f}), "
            f"avg_reward={avg_acc:.4f}, avg_cost={avg_cost:.4f}"
        )



In [5]:
summarize_results(all_results)


Task-Only baseline
Across 3 seeds:
  avg_reward = 0.9356 +- 0.0172
  avg_cost   = 0.2533 +- 0.0157

RC (omega=0.25)
Across 3 seeds:
  avg_reward = 0.9258 +- 0.0141
  avg_cost   = 0.2169 +- 0.0204

RC (omega=0.75)
Across 3 seeds:
  avg_reward = 0.5000 +- 0.0000
  avg_cost   = 0.0000 +- 0.0000
